# Word2Vec – Zadania praktyczne

W tym notebooku samodzielnie zbudujesz model Word2Vec na tekstach polskiej literatury klasycznej.

**Zasady:**
- Uzupełniaj miejsca oznaczone `# TODO`
- Nie modyfikuj kodu poza sekcjami TODO
- Każde zadanie ma testy sprawdzające poprawność

**Dane:** teksty z serwisu Wolne Lektury w folderze `../WOLNE_LEKTURY_SCRAPING/books/`

---
## Zadanie 1 – Wczytaj dane

Wczytaj wszystkie pliki `.txt` z folderu `BOOKS_DIR` (rekurencyjnie, tj. ze wszystkich podfolderów).  
Pomijaj pliki krótsze niż 200 znaków.

In [ ]:
import os
import glob

BOOKS_DIR = "../WOLNE_LEKTURY_SCRAPING/books"

# TODO: Znajdź wszystkie pliki .txt rekurencyjnie w BOOKS_DIR
# Wskazówka: użyj glob.glob z parametrem recursive=True i wzorcem "**/*.txt"
all_files = None  # <-- zastąp tę linię

# TODO: Wczytaj treść plików do listy raw_texts.
# Warunek: plik musi mieć więcej niż 200 znaków.
# Obsłuż wyjątki (try/except) – niektóre pliki mogą mieć inne kodowanie.
raw_texts = []    # <-- uzupełnij

# --- Test ---
assert all_files is not None and len(all_files) > 1000, "Powinieneś znaleźć ponad 1000 plików"
assert len(raw_texts) > 500, f"Powinieneś wczytać ponad 500 tekstów, masz: {len(raw_texts)}"
print(f"OK: znaleziono {len(all_files)} plików, wczytano {len(raw_texts)} tekstów")

---
## Zadanie 2 – Preprocessing

Uzupełnij funkcję `preprocess(text)`, która:
1. Zamienia tekst na **małe litery**
2. Usuwa stopkę Wolnych Lektur (zaczyna się od `-----`)
3. Dzieli tekst na zdania (po `.`, `!`, `?`)
4. Z każdego zdania usuwa wszystko **poza literami** (polskimi i łacińskimi)
5. Dzieli zdanie na tokeny (słowa) i zwraca tylko zdania z **co najmniej 4 tokenami**

In [ ]:
import re

def preprocess(text: str) -> list:
    """
    Zamienia tekst na listę zdań.
    Każde zdanie to lista tokenów (stringów).
    Zwraca tylko zdania z >= 4 tokenami.
    """
    # TODO 2a: zamień na małe litery
    
    # TODO 2b: usuń stopkę Wolnych Lektur (wszystko od "-----" do końca)
    # Wskazówka: re.sub(r'-----.*', '', text, flags=re.DOTALL)
    
    # TODO 2c: podziel na zdania po znakach .!?
    # Wskazówka: re.split(r'[.!?]+',...)
    sentences_raw = []

    sentences = []
    for sent in sentences_raw:
        # TODO 2d: usuń wszystko poza literami (polskimi i łacińskimi) i spacjami
        # Wzorzec do zachowania: [a-ząćęłńóśźż\s]
        cleaned = sent  # <-- zastąp
        
        # TODO 2e: podziel na tokeny i dodaj do sentences jeśli >= 4 tokeny
        pass

    return sentences


# --- Testy ---
test1 = preprocess("Pan Tadeusz wyszedł do lasu. Czy to możliwe? Tak!")
assert isinstance(test1, list), "preprocess musi zwracać listę"
assert all(isinstance(s, list) for s in test1), "Każde zdanie musi być listą"
assert all(isinstance(t, str) for s in test1 for t in s), "Tokeny muszą być stringami"

test2 = preprocess("Krótko.")
assert test2 == [], "Zdania < 4 tokenów powinny być pomijane"

test3 = preprocess("To jest tekst z liczbami 123 i znakami! interpunkcyjnymi, testowy.")
flat = [t for s in test3 for t in s]
assert all(t.isalpha() for t in flat), f"Tokeny nie powinny zawierać cyfr ani znaków spec.: {flat}"

test4 = preprocess("Wolne lektury to super strona. -----\nTo jest stopka.")
flat4 = " ".join(t for s in test4 for t in s)
assert "stopka" not in flat4, "Stopka powinna być usunięta"

print("OK: preprocess działa poprawnie")
print(f"Przykład: preprocess('Pan Tadeusz wyszedł do lasu.') = {preprocess('Pan Tadeusz wyszedł do lasu.')}")

In [ ]:
# Przetwórz wszystkie lektury (ta komórka jest gotowa – uruchom po ukończeniu Zadania 2)
all_sentences = []
for text in raw_texts:
    all_sentences.extend(preprocess(text))

assert len(all_sentences) > 100_000, f"Za mało zdań: {len(all_sentences)}. Sprawdź preprocess."
print(f"Łącznie zdań:   {len(all_sentences):>12,}")
print(f"Łącznie tokenów:{sum(len(s) for s in all_sentences):>12,}")
print(f"Przykładowe zdanie: {all_sentences[42]}")

---
## Zadanie 3 – Trenowanie modelu

Wytrenuj model Word2Vec z podanymi wymaganiami:

| Parametr | Wymagana wartość | Opis |
|---|---|---|
| `vector_size` | 100 | wymiarowość wektora |
| `window` | 5 | rozmiar okna kontekstu |
| `min_count` | 5 | minimalna liczba wystąpień |
| `sg` | 1 | architektura Skip-gram |
| `epochs` | 10 | liczba epok |
| `seed` | 42 | dla reprodukowalności |

In [ ]:
from gensim.models import Word2Vec
import time

# TODO: Wytrenuj model Word2Vec zgodnie z tabelą wyżej.
# Zapisz wynik do zmiennej `model`.
model = None  # <-- zastąp

# --- Test ---
from gensim.models import Word2Vec as _W2V
assert model is not None and isinstance(model, _W2V), "model musi być instancją Word2Vec"
assert model.vector_size == 100,    f"vector_size powinien wynosić 100, masz: {model.vector_size}"
assert model.window == 5,           f"window powinien wynosić 5, masz: {model.window}"
assert model.sg == 1,               f"sg powinien wynosić 1 (Skip-gram), masz: {model.sg}"
assert len(model.wv) > 50_000,      f"Słownik za mały: {len(model.wv)}. Sprawdź min_count i dane."
print(f"OK: model wytrenowany, słownik = {len(model.wv):,} słów")

---
## Zadanie 4 – Podobieństwo semantyczne

### 4a. Uzupełnij funkcję obliczającą podobieństwo cosinusowe **ręcznie** (bez `model.wv.similarity`).

Wzór na podobieństwo cosinusowe:

$$\cos(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \cdot \|\mathbf{b}\|}$$

In [ ]:
import numpy as np

def cosine_similarity(word1: str, word2: str, wv) -> float:
    """
    Oblicza podobieństwo cosinusowe między wektorami dwóch słów.
    Zwraca float w zakresie [-1, 1].
    Jeśli któreś słowo nie istnieje w słowniku, zwraca None.
    """
    if word1 not in wv or word2 not in wv:
        return None

    # TODO 4a: Pobierz wektory obu słów za pomocą wv[word]
    vec_a = None  # <-- zastąp
    vec_b = None  # <-- zastąp

    # TODO 4b: Oblicz iloczyn skalarny (dot product)
    dot = None    # <-- zastąp

    # TODO 4c: Oblicz normy (długości) obu wektorów
    # Wskazówka: np.linalg.norm()
    norm_a = None # <-- zastąp
    norm_b = None # <-- zastąp

    # TODO 4d: Zwróć wynik (uważaj na dzielenie przez zero!)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return None   # <-- zastąp


# --- Test ---
my_sim = cosine_similarity("miłość", "przyjaźń", model.wv)
ref_sim = model.wv.similarity("miłość", "przyjaźń")
assert my_sim is not None, "Funkcja zwróciła None dla istniejących słów"
assert abs(my_sim - ref_sim) < 1e-5, f"Twój wynik ({my_sim:.6f}) różni się od referencyjnego ({ref_sim:.6f})"
assert cosine_similarity("xyzabc", "miłość", model.wv) is None, "Nieznane słowo powinno zwracać None"
print(f"OK: cosine_similarity('miłość', 'przyjaźń') = {my_sim:.4f}")

### 4b. Ranking podobieństwa

Dla każdej pary poniżej oblicz podobieństwo i posortuj od najwyższego do najniższego.  
Wyniki powinny być intuicyjnie sensowne.

In [ ]:
pary = [
    ("miłość",    "nienawiść"),
    ("miłość",    "przyjaźń"),
    ("rzeka",     "morze"),
    ("miecz",     "bitwa"),
    ("chleb",     "poezja"),
    ("ojciec",    "matka"),
    ("król",      "poddany"),
]

# TODO: Oblicz podobieństwo dla każdej pary (użyj swojej funkcji cosine_similarity)
# i wydrukuj wyniki posortowane od najwyższego podobieństwa do najniższego.
# Format: "miłość  ↔  przyjaźń : 0.8310"

# Twój kod tutaj:


---
## Zadanie 5 – Analogie

Uzupełnij funkcję `znajdz_analogie`, która oblicza:

```
A - B + C = ?
```

np. `król - mężczyzna + kobieta = ?`

**Nie możesz** użyć `model.wv.most_similar` – musisz samodzielnie obliczyć wektor wynikowy i przeszukać słownik.

In [ ]:
def znajdz_analogie(A: str, B: str, C: str, wv, topn: int = 5) -> list:
    """
    Oblicza A - B + C i zwraca topn najbliższych słów.
    Wyklucza słowa A, B, C z wyników.
    Zwraca listę krotek: [(słowo, podobieństwo), ...]
    """
    for w in (A, B, C):
        if w not in wv:
            print(f"Słowo '{w}' nie istnieje w słowniku")
            return []

    # TODO 5a: Oblicz wektor wynikowy: vec_A - vec_B + vec_C
    wynik_vec = None  # <-- zastąp

    # TODO 5b: Znormalizuj wektor wynikowy (podziel przez jego normę)
    # Wskazówka: np.linalg.norm()
    
    # TODO 5c: Dla każdego słowa w słowniku (wv.index_to_key) oblicz podobieństwo
    # cosinusowe z wektorem wynikowym i zachowaj topn najlepszych wyników.
    # Wyklucz słowa A, B, C.
    # Wskazówka: wektory całego słownika znajdziesz w wv.vectors (macierz numpy)
    #            wv.index_to_key to lista słów w kolejności indeksów
    
    wyniki = []  # lista (słowo, podobieństwo)
    # Twój kod tutaj:
    
    return sorted(wyniki, key=lambda x: x[1], reverse=True)[:topn]


# --- Test ---
wyniki = znajdz_analogie("król", "mężczyzna", "kobieta", model.wv)
assert len(wyniki) == 5, f"Oczekiwano 5 wyników, dostałem {len(wyniki)}"
assert all(isinstance(w, str) and isinstance(s, float) for w, s in wyniki), "Wyniki muszą być (str, float)"
slowa_wynik = [w for w, _ in wyniki]
assert "mężczyzna" not in slowa_wynik and "król" not in slowa_wynik, "Wyniki nie mogą zawierać A, B, C"
print(f"OK: król - mężczyzna + kobieta = {slowa_wynik[:3]}")

In [ ]:
# Sprawdź kilka analogii
def pokaz_analogie(A, B, C):
    wyniki = znajdz_analogie(A, B, C, model.wv)
    print(f"\n{A} - {B} + {C} = ?")
    for i, (w, s) in enumerate(wyniki, 1):
        print(f"  {i}. {w:20} ({s:.4f})")

pokaz_analogie("król",  "mężczyzna", "kobieta")
pokaz_analogie("dzień", "słońce",    "księżyc")
pokaz_analogie("pies",  "szczekać",  "miauczeć")

---
## Zadanie 6 – Wizualizacja t-SNE

Stwórz wizualizację t-SNE dla **własnych** grup tematycznych.

Wymagania:
- Co najmniej **4 grupy** tematyczne
- Co najmniej **5 słów** w każdej grupie
- Każda grupa narysowana innym kolorem z legendą
- Tytuł wykresu opisujący co pokazuje wizualizacja

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
from sklearn.manifold import TSNE
from matplotlib.patches import Patch

matplotlib.rcParams['font.family'] = 'DejaVu Sans'

# TODO 6a: Zdefiniuj własne grupy tematyczne.
# Klucz = nazwa grupy, wartość = lista słów.
# Sprawdź czy słowa są w słowniku: word in model.wv
moje_grupy = {
    # "przykład": ["słowo1", "słowo2", ...],
}  # <-- uzupełnij


# --- Kod rysujący (gotowy – nie modyfikuj) ---
def rysuj_tsne(grupy, tytul):
    slowa, etykiety = [], []
    for nazwa, lista in grupy.items():
        for s in lista:
            if s in model.wv:
                slowa.append(s)
                etykiety.append(nazwa)

    assert len(slowa) >= 20, f"Za mało słów w słowniku: {len(slowa)}. Dodaj więcej lub zmień słowa."

    wektory = model.wv[slowa]
    tsne = TSNE(n_components=2, perplexity=min(15, len(slowa)-1), random_state=42, max_iter=1000)
    pts = tsne.fit_transform(wektory)

    kolory = plt.cm.tab10.colors
    unikalne = list(grupy.keys())

    fig, ax = plt.subplots(figsize=(13, 8))
    for i, (x, y) in enumerate(pts):
        c = kolory[unikalne.index(etykiety[i]) % len(kolory)]
        ax.scatter(x, y, color=c, s=90, zorder=2)
        ax.annotate(slowa[i], (x, y), fontsize=10, ha='center', va='bottom',
                    xytext=(0, 5), textcoords='offset points')
    ax.legend(handles=[Patch(color=kolory[i % len(kolory)], label=g) for i, g in enumerate(unikalne)])
    ax.set_title(tytul, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# TODO 6b: Wywołaj rysuj_tsne z twoimi grupami i opisowym tytułem
# rysuj_tsne(moje_grupy, "Twój tytuł")

---
## Zadanie 7 – Wpływ parametrów

Wytrenuj **drugi model** z innymi parametrami i porównaj wyniki.

Zmień **co najmniej dwa** parametry spośród: `vector_size`, `window`, `sg`, `epochs`.  
Następnie odpowiedz na pytania w komentarzach.

In [ ]:
# TODO 7a: Wytrenuj model_b z innymi parametrami
model_b = None  # <-- zastąp

# TODO 7b: Dla słów poniżej wydrukuj top-5 podobnych z obu modeli
slowa_test = ["miłość", "bitwa", "bóg"]

for slowo in slowa_test:
    if slowo not in model.wv or slowo not in model_b.wv:
        print(f"Słowo '{slowo}' nie ma w jednym z modeli")
        continue
    print(f"\n{'='*50}")
    print(f"Słowo: {slowo}")
    # TODO: wydrukuj top-5 dla model i model_b obok siebie


# TODO 7c: Odpowiedz na pytania (zamień None na odpowiedź jako string)
odpowiedzi = {
    "Jakie parametry zmieniłeś?": None,
    "Który model daje semantycznie lepsze wyniki i dlaczego?": None,
    "Co się dzieje gdy zwiększysz window?": None,
}

for pytanie, odp in odpowiedzi.items():
    assert odp is not None and len(str(odp)) > 10, f"Odpowiedz na pytanie: '{pytanie}'"
    print(f"\n{pytanie}\n  → {odp}")

---
## Podsumowanie wyników

Uruchom tę komórkę na końcu – sprawdza czy wszystkie zadania są ukończone.

In [ ]:
wyniki = {}

# Zadanie 1
try:
    assert all_files and len(all_files) > 1000 and len(raw_texts) > 500
    wyniki["Zadanie 1 – Wczytanie danych"] = "ZALICZONE"
except:
    wyniki["Zadanie 1 – Wczytanie danych"] = "NIEZALICZONE"

# Zadanie 2
try:
    t = preprocess("Pan Tadeusz szedł przez las i śpiewał. Krótko.")
    assert isinstance(t, list) and len(t) == 1 and len(t[0]) >= 4
    assert len(all_sentences) > 100_000
    wyniki["Zadanie 2 – Preprocessing"] = "ZALICZONE"
except Exception as e:
    wyniki["Zadanie 2 – Preprocessing"] = f"NIEZALICZONE ({e})"

# Zadanie 3
try:
    from gensim.models import Word2Vec as _W
    assert isinstance(model, _W) and model.vector_size == 100 and model.sg == 1
    assert len(model.wv) > 50_000
    wyniki["Zadanie 3 – Trenowanie modelu"] = "ZALICZONE"
except Exception as e:
    wyniki["Zadanie 3 – Trenowanie modelu"] = f"NIEZALICZONE ({e})"

# Zadanie 4
try:
    s = cosine_similarity("miłość", "przyjaźń", model.wv)
    ref = model.wv.similarity("miłość", "przyjaźń")
    assert s is not None and abs(s - ref) < 1e-5
    wyniki["Zadanie 4 – Podobieństwo cosinusowe"] = "ZALICZONE"
except Exception as e:
    wyniki["Zadanie 4 – Podobieństwo cosinusowe"] = f"NIEZALICZONE ({e})"

# Zadanie 5
try:
    r = znajdz_analogie("król", "mężczyzna", "kobieta", model.wv)
    assert len(r) == 5 and "król" not in [w for w,_ in r]
    wyniki["Zadanie 5 – Analogie"] = "ZALICZONE"
except Exception as e:
    wyniki["Zadanie 5 – Analogie"] = f"NIEZALICZONE ({e})"

# Zadanie 6
try:
    assert len(moje_grupy) >= 4
    assert all(len(v) >= 5 for v in moje_grupy.values())
    wyniki["Zadanie 6 – Wizualizacja t-SNE"] = "ZALICZONE"
except Exception as e:
    wyniki["Zadanie 6 – Wizualizacja t-SNE"] = f"NIEZALICZONE ({e})"

# Zadanie 7
try:
    from gensim.models import Word2Vec as _W
    assert isinstance(model_b, _W)
    assert not all(
        getattr(model, p) == getattr(model_b, p)
        for p in ["vector_size", "window", "sg", "epochs"]
    )
    wyniki["Zadanie 7 – Porównanie parametrów"] = "ZALICZONE"
except Exception as e:
    wyniki["Zadanie 7 – Porównanie parametrów"] = f"NIEZALICZONE ({e})"

# Raport
zaliczone = sum(1 for v in wyniki.values() if v == "ZALICZONE")
print("=" * 55)
print(f"  WYNIKI: {zaliczone}/{len(wyniki)} zadań zaliczonych")
print("=" * 55)
for nazwa, status in wyniki.items():
    ikona = "✓" if status == "ZALICZONE" else "✗"
    print(f"  {ikona}  {nazwa:40}  {status}")